# 🎬 MiniMax H3 Studio on Google Colab
### ⚡ A web studio for generating cinematic videos with sound, powered by ComfyUI

This notebook deploys the full **MiniMax H3 Studio** web interface right inside Google Colab, reachable through a public link (**Cloudflare / Ngrok / Localtunnel**).

---

### ⚠️ Important note on GPUs and memory:
**MiniMax H3** is one of the most powerful video generation models (~44 GB of weights in total):
* **Diffusion Model (FastVideo INT8 ConvRot):** ~22.9 GB
* **Text Encoder (Qwen3-VL 32B NVFP4):** ~15.7 GB
* **VAE (Video + Audio):** ~3.2 GB
* **Turbo LoRA & Latent Upscaler:** ~2.6 GB

**📌 Which GPU to pick in the Colab menu:**
* Menu: `Runtime` ➔ `Change runtime type`
* **Recommended:** **NVIDIA L4 (24GB VRAM)** or **A100 (40/80GB VRAM)**. Be sure to enable the **High-RAM** option!
* **Free T4 (15GB VRAM, 12GB RAM):** Because the models are so large, the ~38 GB set will run out of memory (CUDA OOM) on a free T4. The notebook automatically adds 16 GB of swap, but L4 or A100 is recommended for stable generation.

---

### 💾 Saving generation results:
* In **Step 2**, connect Google Drive: all videos are saved automatically to `MyDrive/MiniMax_Studio/outputs` and stay with you even after Colab is closed.
* All generation settings, prompts and seeds are embedded in the MP4 file itself: you can always restore them in the UI with the metadata button!

Run the cells one by one (or press `Ctrl + F9`). The output of Step 7 will show a working link to the web interface!

In [ ]:
#@title 1. 🔍 Check GPU, system memory and set up SWAP
import os
import sys
import json
import torch
import psutil

!nvidia-smi

if not torch.cuda.is_available():
    raise SystemError("❌ No GPU detected! Enable a GPU via 'Runtime' -> 'Change runtime type'.")

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
ram_gb = psutil.virtual_memory().total / (1024**3)

print(f"\n{'='*65}")
print(f"🔍 GPU: {gpu_name}")
print(f"🧠 Available VRAM: {vram_gb:.1f} GB")
print(f"💻 System RAM: {ram_gb:.1f} GB")

if "T4" in gpu_name:
    print("\n⚠️ Warning: Tesla T4 detected (15GB VRAM). The MiniMax H3 model is very heavy.")
    print("💡 Creating a 16 GB swap file on the SSD to avoid out-of-memory crashes...")
    !fallocate -l 16G /swapfile && chmod 600 /swapfile && mkswap /swapfile && swapon /swapfile || true
    !swapon --show
    COMFY_EXTRA_ARGS = "--lowvram --disable-async-offload"
elif "L4" in gpu_name:
    print("\n🚀 NVIDIA L4 (24GB VRAM): the models (~44 GB) do not fit at once, ComfyUI will offload to RAM (slower).")
    COMFY_EXTRA_ARGS = "--reserve-vram 2"
elif "RTX PRO 6000" in gpu_name or "Blackwell" in gpu_name:
    print(f"\n🚀 Blackwell GPU! {gpu_name} ({vram_gb:.0f}GB VRAM) - native NVFP4, all models fit in VRAM.")
    COMFY_EXTRA_ARGS = "--reserve-vram 3 --highvram"
elif any(x in gpu_name for x in ["A100", "H100"]):
    print(f"\n🚀 Top-tier performance! {gpu_name} ({vram_gb:.0f}GB VRAM).")
    COMFY_EXTRA_ARGS = "--reserve-vram 3 --highvram"
else:
    COMFY_EXTRA_ARGS = "--reserve-vram 2"

print(f"⚙️ ComfyUI optimization flags: {COMFY_EXTRA_ARGS}")
print(f"{'='*65}\n")


In [ ]:
#@title 2. 📁 Connect Google Drive (save videos)
#@markdown Connect Google Drive so generated videos (`output`) are saved to your Drive and are not lost when the session disconnects.
import os
import sys

MOUNT_DRIVE = True #@param {type:"boolean"}
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/MiniMax_Studio/outputs"

if MOUNT_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
        print(f"✅ Google Drive connected!\n📁 Video output folder: {DRIVE_OUTPUT_DIR}")
    except Exception as e:
        print(f"⚠️ Could not connect Drive: {e}. Files will be saved locally in the Colab session.")
        MOUNT_DRIVE = False
else:
    print("ℹ️ Google Drive is not connected. Files are saved to local Colab storage.")


In [ ]:
#@title 3. 📦 Install system utilities and Node.js 20 LTS
import os
import subprocess

print("📦 [1/3] Installing system packages (ffmpeg, aria2, ninja-build)...")
!apt-get update -qq > /dev/null 2>&1; apt-get install -y -qq ffmpeg aria2 ninja-build libgl1 libglib2.0-0 > /tmp/apt-install.log 2>&1 || tail -20 /tmp/apt-install.log
missing = [t for t in ("ffmpeg", "aria2c") if subprocess.run(["which", t], capture_output=True).returncode != 0]
if missing:
    raise RuntimeError(f"❌ System packages failed to install: {missing} (see /tmp/apt-install.log)")

print("📦 [2/3] Installing Node.js 20.x LTS from NodeSource...")
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null 2>&1
!apt-get install -y -qq nodejs > /dev/null 2>&1

node_ver = !node -v
npm_ver = !npm -v
print(f"✅ Node.js: {node_ver[0] if node_ver else 'OK'} | npm: {npm_ver[0] if npm_ver else 'OK'}")

print("📦 [3/3] Installing Cloudflared and PyNgrok...")
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
!rm -f cloudflared-linux-amd64.deb
!pip install pyngrok psutil --quiet

print("✅ All system utilities installed!")


In [ ]:
#@title 4. 🧩 Install ComfyUI & custom nodes for MiniMax H3
import os

COMFY_DIR = "/content/ComfyUI"

if not os.path.exists(COMFY_DIR):
    print("📥 Cloning ComfyUI...")
    !git clone https://github.com/comfyanonymous/ComfyUI.git $COMFY_DIR
else:
    print("🔄 ComfyUI is already installed. Checking for updates...")
    %cd $COMFY_DIR
    !git pull

%cd $COMFY_DIR
print("📦 Installing ComfyUI Python dependencies...")
!pip install -r requirements.txt --quiet
!pip install triton xformers torchsde soundfile accelerate sentencepiece --quiet

CUSTOM_NODES_DIR = f"{COMFY_DIR}/custom_nodes"
os.makedirs(CUSTOM_NODES_DIR, exist_ok=True)

# Required custom nodes for MiniMax H3
NODES = [
    ("ComfyUI-MiniMax-H3-SPEED", "https://github.com/StanLukuvka/ComfyUI-MiniMax-H3-SPEED.git"),
    ("ComfyUI-PT_H3ConcatAVLatent", "https://github.com/ptmaster/ComfyUI-PT_H3ConcatAVLatent.git"),
    ("Comfyui_Minimax_h3_latent_Upscaler", "https://github.com/LBH-123-AI/Comfyui_Minimax_h3_latent_Upscaler.git"),
    ("ComfyUI-VideoHelperSuite", "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git"),
    ("ComfyUI-KJNodes", "https://github.com/kijai/ComfyUI-KJNodes.git"),
    ("rgthree-comfy", "https://github.com/rgthree/rgthree-comfy.git"),
    ("ComfyUI-GGUF", "https://github.com/city96/ComfyUI-GGUF.git"),
    ("comfyui-minimax-h3-audio-T8", "https://github.com/T8mars/comfyui-minimax-h3-audio-T8.git"),
]

print("\n🧩 Installing custom nodes for MiniMax H3:")
for name, url in NODES:
    target = f"{CUSTOM_NODES_DIR}/{name}"
    if not os.path.exists(target):
        print(f"  + Cloning {name}...")
        !git clone $url $target --quiet
        req_file = f"{target}/requirements.txt"
        if os.path.exists(req_file):
            !pip install -r $req_file --quiet
    else:
        print(f"  ✓ {name} is already installed.")

# Link the output folder to Google Drive
if 'MOUNT_DRIVE' in globals() and MOUNT_DRIVE and os.path.exists(DRIVE_OUTPUT_DIR):
    !rm -rf $COMFY_DIR/output
    !ln -s $DRIVE_OUTPUT_DIR $COMFY_DIR/output
    print(f"\n🔗 ComfyUI/output folder linked to Google Drive!")
else:
    os.makedirs(f"{COMFY_DIR}/output", exist_ok=True)

os.makedirs(f"{COMFY_DIR}/input", exist_ok=True)
print("\n✅ ComfyUI and custom nodes are ready!")


In [ ]:
#@title 5. 🌐 Install MiniMax Studio Web UI (Frontend)
#@markdown MiniMax Studio GitHub repository:
GITHUB_REPO_URL = "https://github.com/hijal/demo-drama-prototype.git" #@param {type:"string"}
BRANCH = "main" #@param {type:"string"}

import os

STUDIO_DIR = "/content/MiniMax_Studio"

if not os.path.exists(STUDIO_DIR):
    print(f"📥 Cloning {GITHUB_REPO_URL} (branch {BRANCH})...")
    !git clone -b $BRANCH $GITHUB_REPO_URL $STUDIO_DIR
elif os.path.exists(STUDIO_DIR) and os.path.exists(f"{STUDIO_DIR}/.git"):
    print("🔄 Updating the Web UI repository...")
    %cd $STUDIO_DIR
    !git pull

if os.path.exists(STUDIO_DIR):
    %cd $STUDIO_DIR

    # Create the ComfyUI_Root wrapper so Next.js can find ComfyUI/output and ComfyUI/input
    COMFY_WRAPPER = "/content/ComfyUI_Root"
    os.makedirs(COMFY_WRAPPER, exist_ok=True)
    if not os.path.exists(f"{COMFY_WRAPPER}/ComfyUI"):
        !ln -s /content/ComfyUI $COMFY_WRAPPER/ComfyUI

    # Write the .env file
    env_text = f"""COMFY_URL=http://127.0.0.1:8188
COMFY_ROOT={COMFY_WRAPPER}
BIND_HOST=0.0.0.0
PORT=3000
NODE_ENV=production
"""
    with open(f"{STUDIO_DIR}/.env", "w") as f:
        f.write(env_text)
    print("⚙️ .env configuration written.")

    print("📦 Installing npm dependencies (Next.js, Radix, Tailwind)...\n")
    !npm install --no-audit --no-fund --quiet
    # Production build: dev mode blocks requests coming through the tunnel domain
    print("\n🏗️ Building the production bundle (2-4 min)...")
    !npm run build > /tmp/nextjs-build.log 2>&1 || (tail -40 /tmp/nextjs-build.log; exit 1)
    if not os.path.exists(f"{STUDIO_DIR}/.next/BUILD_ID"):
        raise RuntimeError("❌ Web UI build failed - see /tmp/nextjs-build.log")
    print("\n✅ MiniMax Studio Web UI is ready to launch!")
else:
    print(f"ℹ️ Waiting for source files in {STUDIO_DIR}")


In [ ]:
#@title 6. ⏬ Fast download of MiniMax H3 models (aria2c)
#@markdown Tick the models to download (video generation needs all 7):
DOWNLOAD_DIFFUSION = True #@param {type:"boolean"}
#@markdown *Diffusion (FastVideo VSA 1300-step 4-step INT8 ConvRot) — ~22.9 GB*
DOWNLOAD_TEXT_ENCODER = True #@param {type:"boolean"}
#@markdown *Text Encoder (Qwen3VL 32B NVFP4 AWQ) — ~15.7 GB*
DOWNLOAD_VIDEO_VAE = True #@param {type:"boolean"}
#@markdown *Video VAE (INT8 ConvRot) — ~2.6 GB*
DOWNLOAD_AUDIO_VAE = True #@param {type:"boolean"}
#@markdown *Audio VAE (FP32) — ~0.6 GB*
DOWNLOAD_TURBO_LORA = True #@param {type:"boolean"}
#@markdown *Turbo LoRA (4-step v0.1 BF16) — ~1.9 GB*
DOWNLOAD_LATENT_UPSCALER = True #@param {type:"boolean"}
#@markdown *Latent Upscaler 3D (FP16) — ~691 MB*
DOWNLOAD_TAE_PREVIEW = True #@param {type:"boolean"}
#@markdown *TAE Live Preview Decoder — ~9.8 MB*

import os
import sys
import subprocess

MODELS_DIR = "/content/ComfyUI/models"

print("="*65)
print("⏬ CHECKING AND DOWNLOADING MINIMAX H3 MODELS:")
print(f" • Diffusion Model (22.9 GB):    {'ON' if DOWNLOAD_DIFFUSION else 'OFF'}")
print(f" • Text Encoder (15.7 GB):        {'ON' if DOWNLOAD_TEXT_ENCODER else 'OFF'}")
print(f" • Video VAE (2.6 GB):            {'ON' if DOWNLOAD_VIDEO_VAE else 'OFF'}")
print(f" • Audio VAE (0.6 GB):            {'ON' if DOWNLOAD_AUDIO_VAE else 'OFF'}")
print(f" • Turbo LoRA (1.9 GB):           {'ON' if DOWNLOAD_TURBO_LORA else 'OFF'}")
print(f" • Latent Upscaler (0.7 GB):      {'ON' if DOWNLOAD_LATENT_UPSCALER else 'OFF'}")
print(f" • TAE Preview (10 MB):           {'ON' if DOWNLOAD_TAE_PREVIEW else 'OFF'}")
print("="*65 + "\n")

def download_aria2(url, target_dir, filename, min_size_mb=100):
    os.makedirs(target_dir, exist_ok=True)
    full_path = os.path.join(target_dir, filename)
    aria2_file = full_path + ".aria2"
    if os.path.exists(full_path) and not os.path.exists(aria2_file):
        actual_mb = os.path.getsize(full_path) / (1024 * 1024)
        if actual_mb >= min_size_mb:
            print(f"  ✓ {filename} already downloaded ({actual_mb/1024:.2f} GB). Skipping.")
            return
    print(f"\n⏬ Downloading {filename} at high speed...")
    cmd = [
        "aria2c",
        "--console-log-level=warn",
        "-c",
        "-x", "16",
        "-s", "16",
        "-k", "1M",
        "--file-allocation=none",
        "-d", target_dir,
        "-o", filename,
        url
    ]
    subprocess.run(cmd, check=True)
    actual_gb = os.path.getsize(full_path) / (1024**3)
    print(f"  ✅ Done: {filename} ({actual_gb:.2f} GB)")

if DOWNLOAD_DIFFUSION:
    download_aria2(
        "https://huggingface.co/Kijai/MiniMax-H3-experimental/resolve/main/minimax_h3_fastvideo_vsa_datafree_1300step_4step_int8_convrot.safetensors",
        f"{MODELS_DIR}/diffusion_models",
        "minimax_h3_fastvideo_vsa_datafree_1300step_4step_int8_convrot.safetensors",
        min_size_mb=20000
    )

if DOWNLOAD_TEXT_ENCODER:
    download_aria2(
        "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors",
        f"{MODELS_DIR}/text_encoders",
        "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors",
        min_size_mb=14000
    )

if DOWNLOAD_VIDEO_VAE:
    download_aria2(
        "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/vae/minimax_h3_video_vae_int8_convrot.safetensors",
        f"{MODELS_DIR}/vae",
        "minimax_h3_video_vae_int8_convrot.safetensors",
        min_size_mb=2000
    )

if DOWNLOAD_AUDIO_VAE:
    download_aria2(
        "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/vae/minimax_h3_audio_vae_fp32.safetensors",
        f"{MODELS_DIR}/vae",
        "minimax_h3_audio_vae_fp32.safetensors",
        min_size_mb=500
    )

if DOWNLOAD_TURBO_LORA:
    download_aria2(
        "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/loras/minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors",
        f"{MODELS_DIR}/loras",
        "minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors",
        min_size_mb=1500
    )

if DOWNLOAD_LATENT_UPSCALER:
    download_aria2(
        "https://huggingface.co/LBH-123-AI/Minimax_h3_latent_Upscaler/resolve/main/minimax_h3_latent_upscaler_3d_conv_v1/minimax_h3_latent_upscaler_3d_conv_v1_fp16.safetensors",
        f"{MODELS_DIR}/latent_upscale_models",
        "minimax_h3_latent_upscaler_3d_fp16.safetensors",
        min_size_mb=600
    )

if DOWNLOAD_TAE_PREVIEW:
    download_aria2(
        "https://huggingface.co/Kijai/MiniMax-H3-TAE/resolve/main/vae_approx/taeh3.safetensors",
        f"{MODELS_DIR}/vae_approx",
        "taeh3.safetensors",
        min_size_mb=8
    )

print("\n🎉 All selected models downloaded and ready!")


In [ ]:
#@title 7. 🎬 LAUNCH MINIMAX H3 STUDIO (ComfyUI + Web UI + Tunnel)
#@markdown Choose a tunneling service to access the Web UI:
TUNNEL_METHOD = "Cloudflare + Ngrok (Recommended)" #@param ["Cloudflare + Ngrok (Recommended)", "Cloudflare only", "Ngrok only", "All available (Cloudflare + Ngrok + Localtunnel)"]
NGROK_AUTH_TOKEN = "" #@param {type:"string"}
#@markdown 💡 *For Ngrok, copy your token from https://dashboard.ngrok.com/get-started/your-authtoken*
#@markdown ---
#@markdown ⏻ **Auto shutdown:** when no video has been generating for `IDLE_MINUTES`, stop everything and **disconnect & delete the runtime** so compute units stop burning.
AUTO_SHUTDOWN = True #@param {type:"boolean"}
IDLE_MINUTES = 20 #@param {type:"slider", min:5, max:120, step:5}

import subprocess
import time
import os
import re
import urllib.request
import json

COMFY_DIR = "/content/ComfyUI"
STUDIO_DIR = "/content/MiniMax_Studio"

if not os.path.exists(STUDIO_DIR):
    raise FileNotFoundError(f"Folder {STUDIO_DIR} not found. Please run Step 5 first!")

# Kill old processes
!pkill -f "cloudflared" >/dev/null 2>&1 || true
!pkill -f "ngrok" >/dev/null 2>&1 || true
!pkill -f "lt " >/dev/null 2>&1 || true
!pkill -f "main.py" >/dev/null 2>&1 || true
!pkill -f "next dev" >/dev/null 2>&1 || true
!pkill -f "server.js" >/dev/null 2>&1 || true

print("\n" + "="*70)
print("🚀 [1/3] Starting the ComfyUI backend on port 8188...")
comfy_flags = COMFY_EXTRA_ARGS if 'COMFY_EXTRA_ARGS' in globals() else "--reserve-vram 2"
comfy_log = open("/tmp/comfyui.log", "w")
comfy_proc = subprocess.Popen(
    f"python3 main.py --listen 127.0.0.1 --port 8188 --disable-auto-launch {comfy_flags}",
    shell=True,
    cwd=COMFY_DIR,
    stdout=comfy_log,
    stderr=subprocess.STDOUT
)

print("⏳ Waiting for ComfyUI to initialize (usually 10-30 s)...")
ready = False
for _ in range(45):
    try:
        r = urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2)
        if r.status == 200:
            ready = True
            break
    except Exception:
        pass
    time.sleep(2)

if ready:
    print("✅ ComfyUI backend is up and responding!")
else:
    print("⚠️ ComfyUI is still initializing, continuing with the Web UI launch...")

print("\n🚀 [2/3] Starting MiniMax Studio Web UI on port 3000...")
studio_log = open("/tmp/nextjs.log", "w")
studio_proc = subprocess.Popen(
    "PORT=3000 BIND_HOST=0.0.0.0 node server.js",
    shell=True,
    cwd=STUDIO_DIR,
    stdout=studio_log,
    stderr=subprocess.STDOUT
)

print("⏳ Waiting for the Web UI to initialize...")
web_ready = False
for _ in range(30):
    try:
        r = urllib.request.urlopen("http://127.0.0.1:3000", timeout=2)
        if r.status in [200, 307, 308]:
            web_ready = True
            break
    except Exception:
        pass
    time.sleep(2)

if web_ready:
    print("✅ Web UI is ready!")
else:
    print("ℹ️ Web UI is starting in the background...")

print("\n🌐 [3/3] Opening tunnels to the web interface...")
cf_url = None
# Cloudflare always starts, unless only Ngrok is selected
if TUNNEL_METHOD != "Ngrok only":
    try:
        cf_log = open("/tmp/cloudflared.log", "w")
        subprocess.Popen(
            ["cloudflared", "tunnel", "--url", "http://127.0.0.1:3000"],
            stdout=cf_log,
            stderr=subprocess.STDOUT
        )
        for _ in range(20):
            time.sleep(1)
            if os.path.exists("/tmp/cloudflared.log"):
                with open("/tmp/cloudflared.log", "r") as f:
                    log_text = f.read()
                m = re.findall(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', log_text)
                if m:
                    cf_url = m[-1]
                    break
    except Exception as e:
        print(f"⚠️ Cloudflare error: {e}")

ngrok_url = None
if NGROK_AUTH_TOKEN.strip() and TUNNEL_METHOD != "Cloudflare only":
    try:
        from pyngrok import ngrok
        try:
            ngrok.kill()
        except Exception:
            pass
        ngrok.set_auth_token(NGROK_AUTH_TOKEN.strip())
        tunnel = ngrok.connect(3000, "http")
        ngrok_url = tunnel.public_url
    except Exception as e:
        err_str = str(e)
        if "ERR_NGROK_334" in err_str:
            print("\n⚠️ Ngrok: this domain is already used by another running session of yours (ERR_NGROK_334).")
            print("👉 No worries: the Cloudflare link below works fine with no limits!")
        else:
            print(f"\n⚠️ Ngrok error: {e}")

lt_url = None
if (TUNNEL_METHOD == "All available (Cloudflare + Ngrok + Localtunnel)" or (not cf_url and not ngrok_url)):
    try:
        lt_log = open("/tmp/localtunnel.log", "w")
        subprocess.Popen(["npx", "-y", "localtunnel", "--port", "3000"], stdout=lt_log, stderr=subprocess.STDOUT)
        for _ in range(12):
            time.sleep(1)
            if os.path.exists("/tmp/localtunnel.log"):
                with open("/tmp/localtunnel.log", "r") as f:
                    t = f.read()
                m = re.search(r'https://[a-zA-Z0-9-]+\.loca\.lt', t)
                if m:
                    lt_url = m.group(0)
                    break
    except Exception as e:
        print(f"⚠️ Localtunnel error: {e}")

print("\n" + "="*70)
print("🎉 MINIMAX H3 STUDIO WEB INTERFACE IS UP!")
print("="*70)
if cf_url:
    print(f"👉 Cloudflare link: {cf_url}")
if ngrok_url:
    print(f"👉 Ngrok link:      {ngrok_url}")
if lt_url:
    try:
        ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
        print(f"👉 Localtunnel link: {lt_url} (IP password: {ip})")
    except Exception:
        print(f"👉 Localtunnel link: {lt_url}")
if not cf_url and not ngrok_url and not lt_url:
    print("⚠️ Could not read the tunnel link automatically.")
    print("Cloudflare logs: !cat /tmp/cloudflared.log")
print("="*70)
print("💡 Open the link above in your browser to use MiniMax Studio!")
print("To stop, press the Stop button on this cell.\n")

def comfy_busy():
    """True while ComfyUI has a running or pending job."""
    try:
        q = json.loads(urllib.request.urlopen("http://127.0.0.1:8188/queue", timeout=5).read())
        return bool(q.get("queue_running") or q.get("queue_pending"))
    except Exception:
        return False

def shutdown_runtime():
    print("\n🛑 Idle timeout reached - stopping processes and deleting the runtime...")
    comfy_proc.terminate()
    studio_proc.terminate()
    if globals().get("MOUNT_DRIVE"):
        try:
            from google.colab import drive
            drive.flush_and_unmount()  # make sure the last videos are synced to Drive
            print("✅ Google Drive synced.")
        except Exception as e:
            print(f"⚠️ Drive sync failed: {e}")
    from google.colab import runtime
    runtime.unassign()

if AUTO_SHUTDOWN:
    print(f"⏻ Auto shutdown ON: the runtime is deleted after {IDLE_MINUTES} min without generation.")
    if not globals().get("MOUNT_DRIVE"):
        print("⚠️ Google Drive is NOT connected (Step 2) - videos will be LOST when the runtime is deleted. Download them first!")

# Keep the process alive (and watch for idle time)
last_active = time.time()
warned = False
try:
    while True:
        time.sleep(30)
        if not AUTO_SHUTDOWN:
            continue
        if comfy_busy():
            last_active = time.time()
            warned = False
            continue
        idle_min = (time.time() - last_active) / 60
        if not warned and idle_min >= IDLE_MINUTES - 2:
            print(f"⏳ No generation for {idle_min:.0f} min - the runtime will be deleted in ~2 min unless a new generation starts.")
            warned = True
        if idle_min >= IDLE_MINUTES:
            shutdown_runtime()
            break
except KeyboardInterrupt:
    print("\n🛑 Stopping processes...")
    comfy_proc.terminate()
    studio_proc.terminate()
    !pkill -f "cloudflared" >/dev/null 2>&1 || true
    !pkill -f "ngrok" >/dev/null 2>&1 || true
    !pkill -f "lt " >/dev/null 2>&1 || true
    print("Session stopped.")
